# Milestone 2

# Data Splitting

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, train_test_split
from configurations import TRAIN_PATH, TEST_PATH, TARGET, DROP_FEATURES

"""Load data from the paths specified in the configuration"""
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

Train shape: (26729, 10)
Test shape: (11456, 8)


In [2]:
"""
Define the target variable (y) and input features (X).
Drop the columns that will not be used during model training 
(the target itself, the target subtype, and the unique identifier).
"""
y = train_df[TARGET]

drop_cols = DROP_FEATURES + [TARGET]
X = train_df.drop(columns=[col for col in drop_cols if col in train_df.columns])

print("Selected features:", list(X.columns))

Selected features: ['Name', 'DateTime', 'AnimalType', 'SexuponOutcome', 'AgeuponOutcome', 'Breed', 'Color']


In [3]:
"""
Initialize and execute the Stratified K-Fold cross-validation split.
Stratification ensures that the target variable class proportions 
are preserved across all folds.
"""
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

train_df['fold'] = -1

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    train_df.loc[val_idx, 'fold'] = fold_idx

print("Number of rows per fold:")
print(train_df['fold'].value_counts())

Number of rows per fold:
fold
3    5346
2    5346
1    5346
0    5346
4    5345
Name: count, dtype: int64


In [4]:
"""
Check the distribution of the target variable.
Compare the percentage proportions of the target classes between the 
full dataset and fold 0 to ensure the stratified split worked correctly.
"""
fold_0_dist = train_df[train_df['fold'] == 0]['OutcomeType'].value_counts(normalize=True) * 100
total_dist = train_df['OutcomeType'].value_counts(normalize=True) * 100

check_df = pd.DataFrame({'Total %': total_dist, 'Fold 0 %': fold_0_dist})
print(check_df)

                   Total %   Fold 0 %
OutcomeType                          
Adoption         40.289573  40.273101
Transfer         35.250103  35.260007
Return_to_owner  17.905646  17.919940
Euthanasia        5.817651   5.817434
Died              0.737027   0.729517


# Data cleaning

In [5]:
from configurations import TEST_PATH

"""Load the original test dataset."""
test_df = pd.read_csv(TEST_PATH)

print("Missing values in train:")
print(train_df[['Name', 'SexuponOutcome', 'AgeuponOutcome', 'Breed', 'Color']].isnull().sum())

print("\nMissing values in test:")
print(test_df[['Name', 'SexuponOutcome', 'AgeuponOutcome', 'Breed', 'Color']].isnull().sum())

Missing values in train:
Name              7691
SexuponOutcome       1
AgeuponOutcome      18
Breed                0
Color                0
dtype: int64

Missing values in test:
Name              3225
SexuponOutcome       0
AgeuponOutcome       6
Breed                0
Color                0
dtype: int64


In [6]:
MONTH_CONVERSION = {
    'year': 12.0,
    'month': 1.0,
    'week': 7 / 30.4375,
    'day': 1 / 30.4375
}

def parse_age_in_months(age_str):
    """
    Parse a string representing age and convert it to a continuous 
    numerical value in months.

    Args:
        age_str (str): The age string to parse (e.g., '2 years', '3 weeks').

    Returns:
        float: The converted age in months, or np.nan if parsing fails.
    """
    if pd.isna(age_str):
        return np.nan
    parts = str(age_str).split()
    if len(parts) < 2:
        return np.nan

    number = float(parts[0])
    unit = parts[1].lower().rstrip('s')

    multipliyer = MONTH_CONVERSION.get(unit, np.nan)

    return number * multipliyer

"""Apply the conversion function to create a new 'AgeInMonths' numerical column."""
train_df['AgeInMonths'] = train_df['AgeuponOutcome'].apply(parse_age_in_months)
test_df['AgeInMonths'] = test_df['AgeuponOutcome'].apply(parse_age_in_months)

In [7]:
"""
Impute missing values across the datasets.
Categorical columns ('Name', 'SexuponOutcome') are filled with constant placeholders.
The numerical column ('AgeInMonths') is imputed using the median age of the corresponding 'AnimalType'.

To prevent data leakage, the training set imputation is performed fold-by-fold 
using out-of-fold medians. The test set is imputed using overall training set medians.
"""
for df in [train_df, test_df]:
    df['Name'] = df['Name'].fillna('No Name')
    df['SexuponOutcome'] = df['SexuponOutcome'].fillna('Unknown')

for fold in range(5):
    train_fold = train_df[train_df['fold'] != fold]
    val_idx = train_df[train_df['fold'] == fold].index

    type_medians = train_fold.groupby('AnimalType')['AgeInMonths'].median()

    for animal_type, median_val in type_medians.items():
        mask = (train_df.index.isin(val_idx)) & (train_df['AnimalType'] == animal_type) & (train_df['AgeInMonths'].isnull())
        train_df.loc[mask, 'AgeInMonths'] = median_val

overall_medians = train_df.groupby('AnimalType')['AgeInMonths'].median()
for animal_type, median_val in overall_medians.items():
    test_df.loc[(test_df['AnimalType'] == animal_type) & (test_df['AgeInMonths'].isnull()), 'AgeInMonths'] = median_val

In [8]:
"""
Verify that no missing values remain in the specified columns
"""
print("Remaining missing values in train_df:")
print(train_df[['Name', 'SexuponOutcome', 'AgeInMonths']].isnull().sum())

print("\nRemaining missing values in test_df:")
print(test_df[['Name', 'SexuponOutcome', 'AgeInMonths']].isnull().sum())

Remaining missing values in train_df:
Name              0
SexuponOutcome    0
AgeInMonths       0
dtype: int64

Remaining missing values in test_df:
Name              0
SexuponOutcome    0
AgeInMonths       0
dtype: int64


# Data transformation

In [9]:
from sklearn.preprocessing import StandardScaler

"""
Concatenate the cleaned training and test datasets into a single DataFrame (df_all) to ensure consistent 
feature engineering and encoding across both sets.
"""
df_all = pd.concat([train_df, test_df], ignore_index=True)

In [10]:
"""
Create numerical features and scale them:
1. Apply log transformation (log1p) to 'AgeInMonths' to handle skewness.
2. Convert 'DateTime' to a continuous numerical feature ('DaysSinceStart') 
   representing the number of days since the earliest recorded date.
3. Standardize these new numerical features using StandardScaler.
"""
df_all['AgeInMonths_log'] = np.log1p(df_all['AgeInMonths'])

df_all['DateTime'] = pd.to_datetime(df_all['DateTime'])
min_date = df_all['DateTime'].min()
df_all['DaysSinceStart'] = (df_all['DateTime'] - min_date).dt.total_seconds() / (24 * 3600)

scaler = StandardScaler()
df_all[['AgeInMonths_scaled', 'DaysSinceStart_scaled']] = scaler.fit_transform(
    df_all[['AgeInMonths_log', 'DaysSinceStart']]
)

In [11]:
"""
Encode categorical features for machine learning models:
1. Apply One-Hot Encoding to low-cardinality features ('AnimalType', 'SexuponOutcome').
2. Apply Frequency Encoding to high-cardinality features ('Breed', 'Color') 
   to replace categories with their relative frequencies.
3. Create a binary feature ('has_name') indicating if the animal has a specific name.
"""
df_transformed = pd.get_dummies(df_all, columns=['AnimalType', 'SexuponOutcome'], drop_first=True)

for col in ['Breed', 'Color']:
    freq_map = df_all[col].value_counts(normalize=True).to_dict()
    df_transformed[f'{col}_freq'] = df_all[col].map(freq_map)

df_transformed['has_name'] = (df_transformed['Name'] != 'No Name').astype(int)

## Feature Engineering

In [12]:
"""Create new features from the 'DateTime' column to capture temporal patterns."""
df_transformed['DateTime_month'] = df_transformed['DateTime'].dt.month
df_transformed['DateTime_weekday'] = df_transformed['DateTime'].dt.dayofweek
df_transformed['DateTime_hour'] = df_transformed['DateTime'].dt.hour

In [13]:
"""Create a binary feature indicating whether the Breed value contains the word 'Mix'."""
df_transformed['isMix'] = (
    df_transformed['Breed']
    .str.contains(r'\bMix\b', case=False, na=False)
    .astype(int)
)

## Feature Selection

In [14]:
df_transformed.columns

Index(['AnimalID', 'Name', 'DateTime', 'OutcomeType', 'OutcomeSubtype',
       'AgeuponOutcome', 'Breed', 'Color', 'fold', 'AgeInMonths', 'ID',
       'AgeInMonths_log', 'DaysSinceStart', 'AgeInMonths_scaled',
       'DaysSinceStart_scaled', 'AnimalType_Dog', 'SexuponOutcome_Intact Male',
       'SexuponOutcome_Neutered Male', 'SexuponOutcome_Spayed Female',
       'SexuponOutcome_Unknown', 'Breed_freq', 'Color_freq', 'has_name',
       'DateTime_month', 'DateTime_weekday', 'DateTime_hour', 'isMix'],
      dtype='str')

In [ ]:
"""
Drop original and intermediate columns that are no longer needed.
Split the combined dataset back into the original training and test sets 
based on the presence of the target variable ('OutcomeType').
Finally, drop set-specific redundant columns and save the prepared datasets.
"""
cols_to_drop = ['Name', 'DateTime', 'AgeuponOutcome', 'AgeInMonths', 'AgeInMonths_log', 'DaysSinceStart', 'DaysSinceStart_scaled', 'Breed', 'Color', 'AnimalID', 'OutcomeSubtype']
df_final = df_transformed.drop(columns=[c for c in cols_to_drop if c in df_transformed.columns])

train_transformed = df_final[df_final['OutcomeType'].notnull()].copy()
test_transformed = df_final[df_final['OutcomeType'].isnull()].copy()

train_transformed = train_transformed.drop(columns=['ID'], errors='ignore')
test_transformed = test_transformed.drop(columns=['OutcomeType', 'OutcomeSubtype', 'fold'], errors='ignore')

train_transformed.to_csv('train_transformed.csv', index=False)
test_transformed.to_csv('test_transformed.csv', index=False)

print(f"Transformed Train shape: {train_transformed.shape}")
print(f"Transformed Test shape: {test_transformed.shape}")

Transformed Train shape: (26729, 15)
Transformed Test shape: (11456, 14)


## Feature analysis

In [ ]:
import plotly.express as px

# Előkészítés: kiveszük a 'fold' oszlopot a vizsgálatból, ha benne van
df_eda = train_transformed.copy()
if "fold" in df_eda.columns:
  df_eda = df_eda.drop(columns=["fold"])


# ==========================================
# 1. CÉLVÁLTOZÓ SZERINTI ELOSZLÁSOK VIZSGÁLATA
# ==========================================

# A) Életkor (nem skálázott AgeInMonths, ha létezik, különben visszamérjük)
# Használjuk az eredeti hónapokban kifejezett értéket az értelmezhetőség kedvéért:
if "AgeInMonths" in df_eda.columns:
  age_col = "AgeInMonths"
  age_label = "Életkor (hónapokban)"
else:
  age_col = "AgeInMonths_scaled"
  age_label = "Skálázott életkor"

fig1 = px.box(
    df_eda,
    x="OutcomeType",
    y=age_col,
    color="OutcomeType",
    title="Állatok életkora a kimenetel (OutcomeType) függvényében",
    labels={"OutcomeType": "Kimenetel típusa", age_col: age_label},
)
fig1.update_layout(showlegend=False)
fig1.show()


# Oszloponkénti normalizálás (hogy külön látszódjon a keverékek és nem keverékek kimeneteli profilja)
mix_outcome_col_pct = (
    pd.crosstab(
        df_eda["OutcomeType"], df_eda["isMix"], normalize="columns"
    )  # oszloponkénti százalék
    * 100
).reset_index()

# Átalakítjuk Plotly-barát formátumra
mix_melted = mix_outcome_col_pct.melt(
    id_vars="OutcomeType", var_name="isMix", value_name="Percentage"
)
# Átnevezzük a kategóriákat a jobb olvashatóságért
mix_melted["isMix"] = mix_melted["isMix"].map({0: "Nem keverék (0)", 1: "Keverék (1)"})

fig2 = px.bar(
    mix_melted,
    x="OutcomeType",
    y="Percentage",
    color="isMix",
    barmode="group",
    title="Kimenetelek megoszlása külön a tisztavérű és külön a keverék állatok körében (%)",
    labels={
        "OutcomeType": "Kimenetel típusa",
        "Percentage": "Arány a csoporton belül (%)",
        "isMix": "Állat típusa",
    },
    color_discrete_sequence=["#ff9999", "#66b3ff"],
)
fig2.show()


# 1. Kiszámoljuk a soronkénti százalékos megoszlást (OutcomeType-on belül a has_name aránya)
name_outcome_pct = (
    pd.crosstab(df_eda["OutcomeType"], df_eda["has_name"], normalize="index") * 100
).reset_index()

# Átnevezzük az oszlopokat a könnyebb áttekinthetőségért
name_outcome_pct.columns = ["OutcomeType", "Nincs név (0)", "Van név (1)"]

# Átalakítjuk hosszú formátumra (melt) a Plotly számára
name_melted = name_outcome_pct.melt(
    id_vars="OutcomeType", var_name="Név státusz", value_name="Százalék (%)"
)

# 2. Csoportosított oszlopdiagram készítése
fig3 = px.bar(
    name_melted,
    x="OutcomeType",
    y="Százalék (%)",
    color="Név státusz",
    barmode="group",  # Egymás melletti oszlopok a jobb összehasonlíthatóságért
    title=(
        "Név megléte (has_name) megoszlása az egyes kimenetelek"
        " alcsoportjaiban (%)"
    ),
    labels={
        "OutcomeType": "Kimenetel típusa (OutcomeType)",
        "Százalék (%)": "Arány az adott kimenetelen belül (%)",
    },
    color_discrete_sequence=[
        "#ff7f0e",
        "#1f77b4",
    ],  # Narancssárga (nincs név) vs Kék (van név)
)

fig3.update_layout(xaxis_tickangle=-15)
fig3.show()


# ==========================================
# 2. IDŐBELI MINTÁZATOK: Végkimenetelek megoszlása hónapok szerint
# ==========================================
# Azt vizsgáljuk, hogy az egyes hónapokban milyen Outcome-ok fordulnak elő (százalékos megoszlás)
month_outcome = (
    pd.crosstab(
        df_eda["DateTime_month"], df_eda["OutcomeType"], normalize="index"
    )
    * 100
).reset_index()

# Olvashatóbbá tesszük vonaldiagramként
month_outcome_melted = month_outcome.melt(
    id_vars="DateTime_month",
    var_name="OutcomeType",
    value_name="Percentage",
)

fig4 = px.line(
    month_outcome_melted,
    x="DateTime_month",
    y="Percentage",
    color="OutcomeType",
    markers=True,
    title=(
        "Kimenetelek százalékos megoszlása a hónapok függvényében"
        " (Szezonalitás)"
    ),
    labels={
        "DateTime_month": "Hónap",
        "Percentage": "Arány (%)",
        "OutcomeType": "Kimenetel",
    },
)
fig4.show()


# ==========================================
# 3. KORRELÁCIÓS MÁTRIX (A 'fold' nélkül)
# ==========================================
numeric_cols = df_eda.select_dtypes(include=[np.number]).columns
corr_matrix = df_eda[numeric_cols].corr()

fig5 = px.imshow(
    corr_matrix,
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    title="Transzformált jellemzőtér korrelációs mátrixa (Fold nélkül)",
)
fig5.show()